In [ ]:
import stim
from IPython.core.display import Markdown

from library.circuitry import Circuitry
from library.qubit_array import QubitArray
from library.steane_code.compact_patch import SteaneCodePatch
from simulation.noise import NOISELESS
from utils.circuit_faults import analyse_heuristic

In [ ]:
def append_mpp(circuitry: Circuitry, steane: SteaneCodePatch, stage: str, style: str):
    for label, stabilizer in steane.pauli_stabilizers(style).items():
        circuitry.append("MPP", stim.PauliString(stabilizer), tag=NOISELESS)
        qubits.record_measurement(f"{stage}_{label}")
        circuitry.append_tick(tag=NOISELESS)
    circuitry.append(
        "MPP",
        stim.PauliString({q: "Y" for q in steane.support(compact=False)}),
        tag=NOISELESS,
    )
    qubits.record_measurement(f"{stage}_YL")
    circuitry.append_tick(tag=NOISELESS)

In [ ]:
qubits: QubitArray = QubitArray(dimensions=(5, 5))
circuitry = Circuitry(qubits)
steane = SteaneCodePatch(qubits=qubits)

circuitry.annotate_polygons(steane.get_polygons(compact=False))

append_mpp(circuitry, steane, "PREPARATION", style="DIFFUSE")
steane.append_doublecheck(circuitry)
append_mpp(circuitry, steane, "MEASUREMENT", style="DIFFUSE")

# Annotate the X detectors
for label in steane.pauli_stabilizers().keys():
    circuitry.annotate_detector(f"PREPARATION_{label}", f"MEASUREMENT_{label}")

# Annotate the X detectors
for label in ["RZ", "D1", "BZ", "D4", "EX0", "EX1", "Z0", "Z1"]:
    circuitry.annotate_detector(f"STN:CULT:{label}")

# # Annotate the Y measurement detector
circuitry.annotate_detector("PREPARATION_YL", "STN:CULT:CC")
circuitry.annotate_detector("STN:CULT:CC", "STN:CULT:GZ")

circuitry.attach_observable(
    0, "MEASUREMENT_YL", *(f"STN:CULT:{lb}" for lb in ["D1", "D4", "RZ", "GZ"])
)

display(Markdown(f"[Open in Crumble (Cultivation)]({circuitry.to_crumble_url()})"))

circuitry.detectors_report(open_boundaries=True)

In [ ]:
circuit = circuitry.as_stim(per=0.01)
analyse_heuristic(circuit)